In [1]:
import pandas as pd
import random
import re
from transformers import pipeline
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Labeling (Menggunakan model roBERTa)

In [2]:
# 1. Inisialisasi Model Indo-RoBERTa
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

print("Memuat model Indo-RoBERTa... (Ini mungkin memakan waktu beberapa saat)")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name,
    truncation=True,
    max_length=512
)

# 2. Simulasi Data Mentah
try:
    df = pd.read_csv('/content/drive/MyDrive/Kuliah/Semester 6/Jurnal/dataset/komentar_youtube_clean.csv')
    # Pastikan data teks diubah menjadi string dan hilangkan baris yang kosong (NaN)
    df = df.dropna(subset=['clean_text'])
    data_mentah = df['clean_text'].astype(str).tolist()
    print(f"Data berhasil diakses. Total {len(data_mentah)} komentar bersih.")
except FileNotFoundError:
    print("Error: file CSV tidak ditemukan.")
    data_mentah = []
except KeyError:
    print("Error: 'clean_text' kolom tidak ditemukan di file CSV tersebut.")
    data_mentah = []
except Exception as e:
    print(f"Error saat membaca file CSV: {e}")
    data_mentah = []

Memuat model Indo-RoBERTa... (Ini mungkin memakan waktu beberapa saat)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Data berhasil diakses. Total 10990 komentar bersih.


In [9]:
# ==========================================
# 3. PERSIAPAN KAMUS LEKSIKON (ARRAY/LIST)
# ==========================================

positive = [
    "aman", "damai", "stabil", "sejuk", "lindungi", "selamatkan",
    "setuju", "mantap", "hebat", "terbaik", "semangat", "berjaya",
    "menang", "pahlawan", "berani", "keren", "adil", "benar",
    "tulus", "ikhlas", "membela",
]

negative = [
    "perang", "hancur", "blokade", "ancaman", "bahaya", "krisis",
    "mencekam", "darurat", "ketegangan", "mahal", "naik", "susah",
    "sengsara", "antre", "melambung", "penipu", "bohong", "rasis",
    "idiot", "pengecut", "serakah", "sombong", "gila", "munafik",
    "biadab", "kejam", "dzolim", "omon-omon", "ngibul", "konyol",
    "takut", "khawatir", "cemas", "ngeri",
]

neutral = [
    "selat", "hormuz", "wilayah", "perbatasan", "internasional",
    "laut", "teluk", "kawasan", "iran", "amerika", "as", "israel",
    "pbb", "pemerintah", "negara", "presiden", "militer", "rudal",
    "tanker", "minyak", "bbm", "kapal", "ekonomi", "berita",
    "media", "melintas", "lewat", "jalur", "tutup", "buka"
]

In [ ]:
# ==========================================
# 3. PERSIAPAN KAMUS LEKSIKON (ARRAY/LIST)
# ==========================================

positive = [
    "mantap", "terbaik", "hebat", "berani", "cerdas", "maju", "damai", "dukung", "salut", "setuju",
    "benar", "bagus", "kuat"
]

negative = [
     "penipu", "ngibul", "tolol", "rasis", "serakah", "pengecut", "setan", "bodoh",
    "dungu", "goblok", "bego", "idiot", "hancur", "kalah", "gila",
    "munafik", "licik", "kejam"
]

neutral = [
    "iran", "amerika", "indonesia", "israel", "rudal", "kapal",
    "selat", "hormuz", "presiden", "konoha", "minyak", "bbm"
]

In [10]:
# ==========================================
# 4. PROSES HYBRID AUTO-LABELING
# ==========================================
print("Memulai proses hybrid auto-labeling...")

# Asumsi data_mentah sudah ada (misal: data_mentah = df['comment_text'].tolist())
# Untuk contoh ini saya asumsikan data_mentah sudah di-load

# Menyiapkan list berukuran sama dengan data mentah untuk menyimpan hasil
hasil_akhir = [None] * len(data_mentah)

# Variabel untuk menampung data yang tidak ada di kamus dan butuh RoBERTa
teks_untuk_roberta = []
indeks_untuk_roberta = []

# Tahap 1: Cek dengan Kamus Leksikon (Format List)
for i, teks in enumerate(data_mentah):
    teks_lower = str(teks).lower()
    label_ditemukan = None

    # 1. Cek kecocokan dengan kamus POSITIVE
    for kata in positive:
        if re.search(rf"\b{re.escape(kata.lower())}\b", teks_lower):
            label_ditemukan = "Positive"
            break

    # 2. Jika bukan POSITIVE, cek kecocokan dengan kamus NEGATIVE
    if not label_ditemukan:
        for kata in negative:
            if re.search(rf"\b{re.escape(kata.lower())}\b", teks_lower):
                label_ditemukan = "Negative"
                break

    # 3. Jika bukan POSITIVE atau NEGATIVE, cek dengan kamus NEUTRAL
    if not label_ditemukan:
        for kata in neutral:
            if re.search(rf"\b{re.escape(kata.lower())}\b", teks_lower):
                label_ditemukan = "Neutral"
                break

    # 4. Proses penentuan hasil tahap 1
    if label_ditemukan:
        hasil_akhir[i] = {
            "Teks": teks,
            "Label": label_ditemukan,
            "Skor Kepercayaan": 1.0, # Skor penuh karena diambil dari aturan mutlak
            "Sumber Label": "Leksikon"
        }
    else:
        # Jika tidak ada satupun kata yang cocok di ketiga list, baru simpan untuk RoBERTa
        teks_untuk_roberta.append(teks)
        indeks_untuk_roberta.append(i)

# Tahap 2: Proses sisa data dengan RoBERTa
if teks_untuk_roberta:
    print(f"Memproses {len(teks_untuk_roberta)} komentar dengan Indo-RoBERTa (Batching)...")
    # RoBERTa akan memproses dalam batch sekaligus, jauh lebih efisien
    hasil_roberta = sentiment_pipeline(teks_untuk_roberta)

    # Memasukkan hasil RoBERTa kembali ke posisi indeks aslinya
    for i, indeks_asli in enumerate(indeks_untuk_roberta):
        hasil = hasil_roberta[i]
        hasil_akhir[indeks_asli] = {
            "Teks": teks_untuk_roberta[i],
            "Label": hasil['label'].capitalize(),
            "Skor Kepercayaan": round(hasil['score'], 3),
            "Sumber Label": "Indo-RoBERTa"
        }


Memulai proses hybrid auto-labeling...
Memproses 4049 komentar dengan Indo-RoBERTa (Batching)...


In [11]:

# 5. Konversi ke Pandas DataFrame agar rapi
df_hasil = pd.DataFrame(hasil_akhir)

# Menampilkan statistik singkat untuk melihat performa hybrid labeling
print("\n=== Ringkasan Hasil Pelabelan ===")
print(df_hasil['Sumber Label'].value_counts())
print("\n")
print(df_hasil.head())


=== Ringkasan Hasil Pelabelan ===
Sumber Label
Leksikon        6941
Indo-RoBERTa    4049
Name: count, dtype: int64


                                                Teks     Label  \
0  penipu betul perempuan blonde ini sama dengan ...  Negative   
1  tidak boleh diterima????? tanpa rasa malu peny...  Negative   
2            konoha konoha beritanya berbohong terus  Negative   
3  pbb cuma bisa mengecam saja tapi tidak bisa be...   Neutral   
4                                      amerika rasis  Negative   

   Skor Kepercayaan  Sumber Label  
0             1.000      Leksikon  
1             0.996  Indo-RoBERTa  
2             0.998  Indo-RoBERTa  
3             1.000      Leksikon  
4             1.000      Leksikon  


In [12]:
# 5. Evaluasi: Tampilkan 20 Data Secara Random
print("\n" + "="*50)
print("HASIL EVALUASI: 20 DATA RANDOM")
print("="*50)

# Menggunakan pandas .sample() untuk mengambil data acak
# random_state=None agar hasil acaknya berubah-ubah setiap kali dijalankan
df_evaluasi = df_hasil.sample(n=20, random_state=None)

# Menampilkan dataframe dengan format tabel rapi
display(df_evaluasi) # Gunakan print(df_evaluasi.to_string()) jika tidak jalan di Jupyter/Colab


HASIL EVALUASI: 20 DATA RANDOM


,Teks,Label,Skor Kepercayaan,Sumber Label
2544,penyebar propaganda islam fanatic halu,Negative,0.996,Indo-RoBERTa
4376,setuju!,Positive,1.000,Leksikon
1200,tidak mikir dia,Negative,0.996,Indo-RoBERTa
2069,harga yang harus dibayar rakyat indonesia gara...,Negative,0.998,Indo-RoBERTa
8784,banyak alasan untuk terus berperang masih ada ...,Negative,0.976,Indo-RoBERTa
8758,itu ggal di rudal saja kapal amerika kok repot,Neutral,1.000,Leksikon
7247,betul tidak import bbm karena tidak punya uang...,Neutral,1.000,Leksikon
10419,memang babi sih amerika malah dia blokade kapa...,Negative,1.000,Leksikon
8750,as sudah jadi lanun bajak laut di selat hormus...,Neutral,1.000,Leksikon
4396,iran my favorite bravo iran,Neutral,1.000,Leksikon


In [14]:
print("\n" + "="*50)
print("PERSEBARAN LABELING DATA")
print("="*50)

# Menghitung persebaran label
label_distribution = df_hasil['Label'].value_counts()

# Menghitung persentase
total_labels = len(df_hasil)
label_percentages = (label_distribution / total_labels) * 100

# Menggabungkan hasil ke dalam DataFrame untuk tampilan yang rapi
df_label_distribution = pd.DataFrame({
    'Jumlah': label_distribution,
    'Persentase': label_percentages.round(2).astype(str) + '%'
})

print(df_label_distribution)


PERSEBARAN LABELING DATA
          Jumlah Persentase
Label                      
Neutral     4811     43.78%
Negative    4674     42.53%
Positive    1505     13.69%


In [15]:
df_hasil.to_csv('/content/drive/MyDrive/Kuliah/Semester 6/Jurnal/dataset/Komentar_Youtube_bersih_berlabel_3.csv', index=False)